# Attention in Practice with PyTorch

**Module 06: Attention Mechanism**  
**Objective**: Apply attention to real-world sequence tasks

Practical applications:
- Neural machine translation (seq2seq with attention)
- Image captioning
- Question answering
- Text summarization

## What You'll Learn
1. Seq2Seq architecture with attention
2. Implementing Bahdanau attention in PyTorch
3. Translation task (English→French)
4. Attention visualization
5. Beam search decoding

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import string

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}\n")

torch.manual_seed(42)

## 1. Seq2Seq with Attention

**Traditional Seq2Seq**:
```
Encoder → Context Vector (fixed size) → Decoder
```

**With Attention**:
```
Encoder → All hidden states → Attention → Dynamic context → Decoder
```

**Key improvement**: Decoder can focus on relevant parts of input

In [ ]:
class Encoder(nn.Module):
    """RNN encoder."""
    
    def __init__(self, input_dim, embedding_dim, hidden_dim, num_layers=1, dropout=0.2):
        super().__init__()
        
        self.embedding = nn.Embedding(input_dim, embedding_dim)
        self.rnn = nn.LSTM(embedding_dim, hidden_dim, num_layers,
                          batch_first=True, dropout=dropout if num_layers > 1 else 0)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, src):
        """
        Args:
            src: (batch, src_len) source indices
        
        Returns:
            outputs: (batch, src_len, hidden_dim)
            hidden: (num_layers, batch, hidden_dim)
            cell: (num_layers, batch, hidden_dim)
        """
        embedded = self.dropout(self.embedding(src))  # (batch, src_len, emb_dim)
        outputs, (hidden, cell) = self.rnn(embedded)
        
        return outputs, hidden, cell

class BahdanauAttention(nn.Module):
    """Bahdanau attention mechanism."""
    
    def __init__(self, hidden_dim, attention_dim):
        super().__init__()
        
        self.W1 = nn.Linear(hidden_dim, attention_dim, bias=False)  # For decoder
        self.W2 = nn.Linear(hidden_dim, attention_dim, bias=False)  # For encoder
        self.V = nn.Linear(attention_dim, 1, bias=False)
    
    def forward(self, decoder_hidden, encoder_outputs):
        """
        Args:
            decoder_hidden: (batch, hidden_dim) current decoder state
            encoder_outputs: (batch, src_len, hidden_dim) all encoder states
        
        Returns:
            attention_weights: (batch, src_len)
            context: (batch, hidden_dim)
        """
        batch_size, src_len, _ = encoder_outputs.size()
        
        # Expand decoder hidden to match encoder shape
        decoder_hidden = decoder_hidden.unsqueeze(1).repeat(1, src_len, 1)  # (batch, src_len, hidden)
        
        # Compute alignment scores
        # e = v^T * tanh(W1 * decoder + W2 * encoder)
        energy = torch.tanh(self.W1(decoder_hidden) + self.W2(encoder_outputs))  # (batch, src_len, attn_dim)
        attention_scores = self.V(energy).squeeze(2)  # (batch, src_len)
        
        # Normalize with softmax
        attention_weights = F.softmax(attention_scores, dim=1)  # (batch, src_len)
        
        # Context vector (weighted sum of encoder outputs)
        context = torch.bmm(attention_weights.unsqueeze(1), encoder_outputs).squeeze(1)  # (batch, hidden)
        
        return attention_weights, context

class Decoder(nn.Module):
    """RNN decoder with attention."""
    
    def __init__(self, output_dim, embedding_dim, hidden_dim, attention_dim, num_layers=1, dropout=0.2):
        super().__init__()
        
        self.output_dim = output_dim
        self.embedding = nn.Embedding(output_dim, embedding_dim)
        
        self.attention = BahdanauAttention(hidden_dim, attention_dim)
        
        # RNN input = embedding + context
        self.rnn = nn.LSTM(embedding_dim + hidden_dim, hidden_dim, num_layers,
                          batch_first=True, dropout=dropout if num_layers > 1 else 0)
        
        # Output layer
        self.fc = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, target, decoder_hidden, decoder_cell, encoder_outputs):
        """
        Single decoding step.
        
        Args:
            target: (batch,) target token
            decoder_hidden, decoder_cell: decoder state
            encoder_outputs: (batch, src_len, hidden_dim)
        
        Returns:
            prediction: (batch, output_dim)
            decoder_hidden, decoder_cell: updated state
            attention_weights: (batch, src_len)
        """
        target = target.unsqueeze(1)  # (batch, 1)
        embedded = self.dropout(self.embedding(target))  # (batch, 1, emb_dim)
        
        # Compute attention using previous decoder hidden state
        attention_weights, context = self.attention(decoder_hidden[-1], encoder_outputs)
        
        # Concatenate embedded input and context
        rnn_input = torch.cat([embedded, context.unsqueeze(1)], dim=2)  # (batch, 1, emb+hidden)
        
        # Decode
        output, (decoder_hidden, decoder_cell) = self.rnn(rnn_input, (decoder_hidden, decoder_cell))
        
        # Predict
        prediction = self.fc(output.squeeze(1))  # (batch, output_dim)
        
        return prediction, decoder_hidden, decoder_cell, attention_weights

class Seq2Seq(nn.Module):
    """Sequence-to-sequence model with attention."""
    
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
    
    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        """
        Args:
            src: (batch, src_len)
            trg: (batch, trg_len)
            teacher_forcing_ratio: probability of using teacher forcing
        
        Returns:
            outputs: (batch, trg_len, output_dim)
            attention_weights_all: list of attention weights
        """
        batch_size = src.size(0)
        trg_len = trg.size(1)
        trg_vocab_size = self.decoder.output_dim
        
        # Encode
        encoder_outputs, hidden, cell = self.encoder(src)
        
        # First decoder input is <SOS>
        decoder_input = trg[:, 0]
        decoder_hidden, decoder_cell = hidden, cell
        
        outputs = []
        attention_weights_all = []
        
        # Decode
        for t in range(1, trg_len):
            prediction, decoder_hidden, decoder_cell, attention_weights = self.decoder(
                decoder_input, decoder_hidden, decoder_cell, encoder_outputs
            )
            
            outputs.append(prediction.unsqueeze(1))
            attention_weights_all.append(attention_weights)
            
            # Teacher forcing
            teacher_force = np.random.random() < teacher_forcing_ratio
            top1 = prediction.argmax(1)
            decoder_input = trg[:, t] if teacher_force else top1
        
        outputs = torch.cat(outputs, dim=1)  # (batch, trg_len-1, vocab_size)
        
        return outputs, attention_weights_all

# Test architecture
print("Testing Seq2Seq with Attention\n")

INPUT_DIM = 100
OUTPUT_DIM = 80
ENC_EMB_DIM = 64
DEC_EMB_DIM = 64
HIDDEN_DIM = 128
ATTENTION_DIM = 64

encoder = Encoder(INPUT_DIM, ENC_EMB_DIM, HIDDEN_DIM, num_layers=2).to(device)
decoder = Decoder(OUTPUT_DIM, DEC_EMB_DIM, HIDDEN_DIM, ATTENTION_DIM, num_layers=2).to(device)
model = Seq2Seq(encoder, decoder).to(device)

# Test forward
src = torch.randint(0, INPUT_DIM, (4, 10)).to(device)
trg = torch.randint(0, OUTPUT_DIM, (4, 8)).to(device)

outputs, attention = model(src, trg)

print(f"Source shape: {src.shape}")
print(f"Target shape: {trg.shape}")
print(f"Output shape: {outputs.shape}")
print(f"Number of attention matrices: {len(attention)}")
print(f"Attention matrix shape: {attention[0].shape}\n")

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

## 2. Translation Task (Simplified)

**Task**: English → French translation

**Dataset**: Simplified parallel corpus

In [ ]:
# Simplified English-French parallel corpus
parallel_corpus = [
    ("i love you", "je t aime"),
    ("i am happy", "je suis heureux"),
    ("hello world", "bonjour le monde"),
    ("good morning", "bonjour"),
    ("thank you", "merci"),
    ("how are you", "comment allez vous"),
    ("i am fine", "je vais bien"),
    ("good night", "bonne nuit"),
    ("see you later", "a bientot"),
    ("i like this", "j aime ca"),
] * 20  # Repeat for training

print(f"Parallel corpus size: {len(parallel_corpus)}")
print(f"\nSample pairs:")
for i in range(3):
    eng, fra = parallel_corpus[i]
    print(f"  EN: {eng}")
    print(f"  FR: {fra}\n")

# Build vocabularies
def build_vocab(sentences):
    vocab = {'<PAD>': 0, '<SOS>': 1, '<EOS>': 2, '<UNK>': 3}
    for sentence in sentences:
        for word in sentence.split():
            if word not in vocab:
                vocab[word] = len(vocab)
    return vocab

eng_sentences = [pair[0] for pair in parallel_corpus]
fra_sentences = [pair[1] for pair in parallel_corpus]

eng_vocab = build_vocab(eng_sentences)
fra_vocab = build_vocab(fra_sentences)

print(f"English vocab size: {len(eng_vocab)}")
print(f"French vocab size: {len(fra_vocab)}")

# Create reverse vocab
eng_idx2word = {idx: word for word, idx in eng_vocab.items()}
fra_idx2word = {idx: word for word, idx in fra_vocab.items()}

def sentence_to_indices(sentence, vocab, max_len=15):
    """Convert sentence to indices with SOS/EOS."""
    indices = [vocab['<SOS>']]
    for word in sentence.split():
        indices.append(vocab.get(word, vocab['<UNK>']))
    indices.append(vocab['<EOS>'])
    
    # Pad
    while len(indices) < max_len:
        indices.append(vocab['<PAD>'])
    
    return indices[:max_len]

# Prepare dataset
class TranslationDataset(Dataset):
    def __init__(self, pairs, src_vocab, trg_vocab, max_len=15):
        self.pairs = pairs
        self.src_vocab = src_vocab
        self.trg_vocab = trg_vocab
        self.max_len = max_len
    
    def __len__(self):
        return len(self.pairs)
    
    def __getitem__(self, idx):
        src_sent, trg_sent = self.pairs[idx]
        src = torch.LongTensor(sentence_to_indices(src_sent, self.src_vocab, self.max_len))
        trg = torch.LongTensor(sentence_to_indices(trg_sent, self.trg_vocab, self.max_len))
        return src, trg

dataset = TranslationDataset(parallel_corpus, eng_vocab, fra_vocab)
train_loader = DataLoader(dataset, batch_size=32, shuffle=True)

print(f"\nDataset size: {len(dataset)}")
print(f"Batches: {len(train_loader)}")

In [ ]:
# Create translation model
encoder_model = Encoder(len(eng_vocab), embedding_dim=64, hidden_dim=128, num_layers=2, dropout=0.3).to(device)
decoder_model = Decoder(len(fra_vocab), embedding_dim=64, hidden_dim=128, attention_dim=64, num_layers=2, dropout=0.3).to(device)
translation_model = Seq2Seq(encoder_model, decoder_model).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=fra_vocab['<PAD>'])
optimizer = optim.Adam(translation_model.parameters(), lr=0.001)

print(f"Translation model created")
print(f"Parameters: {sum(p.numel() for p in translation_model.parameters()):,}\n")

# Training loop
def train_translation(model, train_loader, epochs=50):
    model.train()
    losses = []
    
    for epoch in range(epochs):
        epoch_loss = 0
        
        for src, trg in train_loader:
            src, trg = src.to(device), trg.to(device)
            
            optimizer.zero_grad()
            
            # Forward
            outputs, _ = model(src, trg, teacher_forcing_ratio=0.5)
            
            # Reshape for loss
            output_dim = outputs.shape[-1]
            outputs = outputs.reshape(-1, output_dim)
            trg = trg[:, 1:].reshape(-1)  # Skip <SOS>
            
            loss = criterion(outputs, trg)
            loss.backward()
            
            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            
            optimizer.step()
            
            epoch_loss += loss.item()
        
        avg_loss = epoch_loss / len(train_loader)
        losses.append(avg_loss)
        
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}/{epochs}: Loss = {avg_loss:.4f}")
    
    return losses

print("Training Translation Model...\n")
losses = train_translation(translation_model, train_loader, epochs=50)

# Plot
plt.figure(figsize=(12, 4))
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Translation Model Training Loss')
plt.grid(True, alpha=0.3)
plt.show()

## 3. Translation and Visualization

In [ ]:
def translate(model, sentence, src_vocab, trg_vocab, max_len=15):
    """Translate a sentence and return attention weights."""
    model.eval()
    
    # Prepare input
    src_indices = sentence_to_indices(sentence, src_vocab, max_len)
    src_tensor = torch.LongTensor(src_indices).unsqueeze(0).to(device)
    
    with torch.no_grad():
        # Encode
        encoder_outputs, hidden, cell = model.encoder(src_tensor)
        
        # Decode
        decoder_input = torch.LongTensor([trg_vocab['<SOS>']]).to(device)
        decoder_hidden, decoder_cell = hidden, cell
        
        translated = []
        attention_weights = []
        
        for _ in range(max_len):
            prediction, decoder_hidden, decoder_cell, attn = model.decoder(
                decoder_input, decoder_hidden, decoder_cell, encoder_outputs
            )
            
            top1 = prediction.argmax(1)
            translated.append(top1.item())
            attention_weights.append(attn.cpu().numpy())
            
            if top1.item() == trg_vocab['<EOS>']:
                break
            
            decoder_input = top1
    
    return translated, attention_weights

# Test translation
test_sentences = [
    "i love you",
    "good morning",
    "how are you",
    "thank you",
]

print("\nTranslation Results:\n")
for sentence in test_sentences:
    translated_indices, attention = translate(translation_model, sentence, eng_vocab, fra_vocab)
    
    translated_words = [fra_idx2word.get(idx, '<UNK>') for idx in translated_indices]
    translated_words = [w for w in translated_words if w not in ['<EOS>', '<PAD>']]
    
    print(f"EN: {sentence}")
    print(f"FR: {' '.join(translated_words)}\n")

In [ ]:
def visualize_attention(sentence, src_vocab, trg_vocab, model):
    """Visualize attention weights."""
    translated_indices, attention_weights = translate(model, sentence, src_vocab, trg_vocab)
    
    # Get words
    src_words = sentence.split() + ['<EOS>']
    trg_words = [fra_idx2word.get(idx, '<UNK>') for idx in translated_indices]
    
    # Stack attention
    if len(attention_weights) > 0:
        attention_matrix = np.vstack([attn[0, :len(src_words)] for attn in attention_weights])
        
        plt.figure(figsize=(10, 6))
        sns.heatmap(attention_matrix, annot=True, fmt='.2f', cmap='Blues',
                    xticklabels=src_words, yticklabels=trg_words,
                    cbar_kws={'label': 'Attention Weight'})
        plt.xlabel('Source (English)')
        plt.ylabel('Target (French)')
        plt.title(f'Attention Visualization: "{sentence}"')
        plt.tight_layout()
        plt.show()

# Visualize
visualize_attention("i love you", eng_vocab, fra_vocab, translation_model)
visualize_attention("how are you", eng_vocab, fra_vocab, translation_model)

## 4. Beam Search

**Greedy decoding**: Choose most probable token at each step

**Problem**: Locally optimal ≠ globally optimal

**Beam search**: Keep top-k hypotheses at each step

**Benefits**: Better translations, controlled via beam size

In [ ]:
def beam_search_decode(model, src, src_vocab, trg_vocab, beam_size=3, max_len=15):
    """Beam search decoder."""
    model.eval()
    
    src_indices = sentence_to_indices(src, src_vocab, max_len)
    src_tensor = torch.LongTensor(src_indices).unsqueeze(0).to(device)
    
    with torch.no_grad():
        encoder_outputs, hidden, cell = model.encoder(src_tensor)
        
        # Initialize beam
        beams = [(torch.LongTensor([trg_vocab['<SOS>']]).to(device), 0.0, hidden, cell)]
        
        for _ in range(max_len):
            candidates = []
            
            for seq, score, h, c in beams:
                if seq[-1].item() == trg_vocab['<EOS>']:
                    candidates.append((seq, score, h, c))
                    continue
                
                prediction, new_h, new_c, _ = model.decoder(
                    seq[-1].unsqueeze(0), h, c, encoder_outputs
                )
                
                log_probs = F.log_softmax(prediction, dim=1)
                top_k = torch.topk(log_probs, beam_size)
                
                for k in range(beam_size):
                    token = top_k.indices[0, k]
                    token_score = top_k.values[0, k].item()
                    
                    new_seq = torch.cat([seq, token.unsqueeze(0)])
                    new_score = score + token_score
                    
                    candidates.append((new_seq, new_score, new_h, new_c))
            
            # Select top beams
            candidates = sorted(candidates, key=lambda x: x[1], reverse=True)[:beam_size]
            beams = candidates
            
            # Check if all beams ended
            if all(seq[-1].item() == trg_vocab['<EOS>'] for seq, _, _, _ in beams):
                break
        
        # Return best beam
        best_seq = beams[0][0].cpu().numpy()
        return best_seq

# Test beam search
print("\nBeam Search Results:\n")
for sentence in test_sentences:
    best_translation = beam_search_decode(translation_model, sentence, eng_vocab, fra_vocab, beam_size=3)
    
    translated_words = [fra_idx2word.get(idx, '<UNK>') for idx in best_translation if idx not in [fra_vocab['<PAD>'], fra_vocab['<SOS>'], fra_vocab['<EOS>']]]
    
    print(f"EN: {sentence}")
    print(f"FR (beam=3): {' '.join(translated_words)}\n")

## Summary

You've mastered attention in practice:
- ✅ Seq2Seq with Bahdanau attention
- ✅ Neural machine translation
- ✅ Attention visualization
- ✅ Beam search decoding

**Key insights**:
1. Attention solves information bottleneck in seq2seq
2. Dynamic context improves translation quality
3. Attention weights show model's focus
4. Beam search finds better global solutions
5. Teacher forcing stabilizes training

**Attention applications**:
- Neural machine translation
- Image captioning (CNN encoder + RNN decoder)
- Text summarization
- Question answering
- Speech recognition

**Next Module**: Transformers (attention is all you need!)

## Exercises

1. **Bidirectional Encoder**: Use bidirectional LSTM in encoder
2. **Copy Mechanism**: Add copy mechanism for handling rare words
3. **Image Captioning**: Replace text encoder with CNN for image→text